# Conformal tetrahedral meshing of a polycrystal

Meshes a voxelised polycrystal sample into grain-conforming tetrahedra: cleaned voxel labels, smoothed grain-boundary surfaces, a closed and validated surface complex, one tetrahedral mesh per grain, quality improvement and an Abaqus export. The notebook ends with element quality histograms (surface triangles and tetrahedra) and a clipped 3D view of the tetrahedra.

This version uses the d3v2p1 speed-ups. Stages with a d3v2p1 version are imported from `upxo.meshing.gbconformant.d3v2p1` in their cell; every other stage is imported from `d3v2p0`.

| Stage | Module | d3v2p1 change |
|---|---|---|
| Grain-interface and joint Gmsh remesh | `gmsh_interfaces`, `gmsh_closed` | parallel intersection checks, faster chart import; same output |
| Pre-tet surface validation | `tet_validation` | parallel per-grain checks, numba intersection search; same report |
| Surface angle repair | `surface_angles` | parallel candidate evaluation in rounds; same rules, order differs |
| Tet meshing | `gmsh_tets` | one Gmsh model per grain, grains in parallel |
| Tet dihedral smoothing | `tet_smoothing` | 8-colour block phases in parallel, numba trial and surface-check kernels; same rules |
| Tet face and edge swaps | `tet_swaps` | numba acceptance test per candidate; same swaps |

Every d3v2p1 function takes `backend` (default `'auto'`) and `n_workers` (default `None`: automatic, the physical cores); without worker processes or numba it runs serially in pure numpy. The tier used is recorded in each stage report. Outputs go to `runs/CM04_dev_d3v2p1`.

Author: Dr. Sunil Anandatheertha

## Overview

Run the cells in order in a fresh kernel. Set `SAMPLE` in the first code cell to the full path of your `.npy` label array (a 3D array of integer grain IDs), and `RUN_DIR` to the folder for the results. The sample is cleaned at voxel level (small grains, spikes and pinches; thin grains can be frozen as voxel facets so they do not collapse during smoothing), then its grain-boundary interfaces are smoothed, remeshed with Gmsh, closed with RVE faces, validated and repaired before tetrahedra are generated and improved. Results are written to `runs/CM04_dev_d3v2p1`; names ending in `_OUTPUT` or `_AUDIT` hold the saved-file locations.

Plots are off by default. The two exceptions are at the end of the notebook: the element quality histograms and the clipped 3D view of the tetrahedra.

## Control guide

These notes describe the existing controls. After changing an upstream setting, rerun its cell and all dependent cells below it. Assignments such as `CAP_MESH_SIZE = GMSH_SIZE` take effect when that cell runs; they are not dynamic links. Physical lengths use the units of `SPACING`. Increasing a limit helps only if that limit is actually preventing progress.

### Input and voxel cleaning

| Control | Meaning; effect of increasing or decreasing |
|---|---|
| `SAMPLE`, `RUN_DIR` | Full path of the input `.npy` label array, and the folder that receives all results. Both are set in the first code cell; give full paths to use your own data. Changing the sample changes the structure. |
| `GRAIN_ID_SEED` | Reproducible random ID/colour permutation. Different integers give different permutations; larger is not necessarily more contrasting. Geometry is unchanged. |
| `CLEAN_SPIKES` | `True` enables UPXO spike removal before refinement; `False` bypasses it. |
| `SPIKE_MAX_PASSES` | Higher allows more cleaning passes; lower stops sooner. No-change and repeated-configuration stops still apply. More passes cannot override protected last voxels. |
| `CLEAN_PINCHES` | Enables targeted voxel contact cleaning; `False` bypasses it. |
| `PINCH_MAX_PASSES` | Higher allows more search passes; lower limits work. Other budgets and acceptance rules still apply. |
| `PINCH_MAX_CHANGES` | Higher allows more accepted voxel reassignments in total; lower is more conservative. |
| `PINCH_MAX_VOLUME_FRACTION` | Higher permits larger per-grain net voxel-count changes; lower preserves counts more tightly. Positive values allow at least one voxel; zero allows none. This is not a displacement limit or total-edit fraction. |
| `PINCH_PROTECT_GRAINS_UP_TO` | Higher protects more small grains from donating voxels; lower permits cleaning smaller grains. Counts are measured after spike cleaning, before upscaling. |
| `UPSCALE_FACTOR` | Integer ≥1: 1 leaves resolution unchanged; higher subdivides along all axes. Memory grows approximately with factor³. Refinement alone preserves the original geometry and contacts; it adds no physical detail. |
| `SPACING` | Original voxel dimensions x/y/z. Increasing a component enlarges that physical dimension; decreasing shrinks it. Use measured spacing, not this setting as a repair control. |

`effective_spacing` is derived from `SPACING / UPSCALE_FACTOR`. Settings using `min(SPACING)` remain tied to original physical voxel size. The pinch detector searches local 2×2×2 face-disconnected arrangements. Its budgets cannot extend detection to every thin neck or later smoothing-induced defect. Accepted edits preserve global six-connected component counts and obey volume/protection rules.

### Surface and junction smoothing

| Control | Increase / decrease effect |
|---|---|
| `LAPLACE_ITERATIONS` | More surface-smoothing steps can reduce staircasing but increase shrinkage, distortion and runtime; fewer retain more voxel detail. Zero disables these steps. |
| `LAPLACE_RELAXATION` | Larger values give stronger averaging steps; smaller are gentler. Use the supported range (0,1]. Local safeguards can suppress movement. |
| `JUNCTION_ITERATIONS` | More steps smooth eligible junction lines further; fewer retain more original shape. Zero disables line smoothing. |
| `JUNCTION_RELAXATION` | Larger values give stronger line-smoothing steps; smaller are gentler. Range (0,1]. |
| `JUNCTION_MAX_DISPLACEMENT` | Larger physical bound permits more total junction movement from extracted geometry; smaller preserves shape more tightly. Extra iterations may do little once constrained. |

Junction endpoints, branches and four-or-more-grain nodes remain frozen, and RVE-plane coordinates stay fixed. Stronger smoothing does not remove these topology constraints and can bring neighbouring sheets closer. Local collapse/reversal safeguards do not certify freedom from global intersections.

### Pre-Gmsh point-contact separation

| Control | Increase / decrease or switch effect |
|---|---|
| `SEPARATE_PINCHES` | Enables splitting disconnected triangle fans at a shared vertex; `False` bypasses it. |
| `PINCH_PRESS_RADIUS` | Larger physical radius affects a broader neighbourhood; smaller localizes pressing near the split tip. |
| `PINCH_PRESS_MAX_DISPLACEMENT` | Larger allows more movement; smaller better preserves shape but may prevent separation. |
| `PINCH_PRESS_ITERATIONS` | More steps permit further adjustment; fewer reduce work and deformation. Constraints still apply. |
| `PINCH_PRESS_RELAXATION` | Larger gives stronger steps; smaller gives gentler adjustment. |
| `PINCH_MIN_OPENING` | Larger demands a wider minimum separation of duplicated tips and can cause rollback if unattainable; smaller accepts narrower separation. It is not a global clearance guarantee. |

Length controls use physical units. This operation handles disconnected point fans: it does **not** cut shared edges or connected multi-grain junctions. Increasing its controls cannot remove contacts outside that scope. Other frozen junction points and RVE coordinates remain constrained.

### Gmsh and RVE caps

| Control | Increase / decrease effect |
|---|---|
| `GMSH_SIZE` | Smaller requests finer surface triangles and greater cost; larger requests coarser triangles. Retained curves and chart seams constrain size. Refinement does not itself remove contacts or overlaps. |
| `GMSH_ALGORITHM` | Categorical: 1 MeshAdapt, 2 Automatic, 5 Delaunay, 6 Frontal-Delaunay. Higher numbers are not inherently better. |
| `CAP_MESH_SIZE` | Smaller requests finer new cap triangles; larger requests coarser ones. |
| `RVE_EDGE_SIZE` | Smaller subdivides new box-edge segments more finely; larger permits longer segments. Existing interface traces remain constrained. |
| `FINAL_GMSH_SIZE` | Same size tradeoff, applied to joint remeshing of internal interfaces and caps. |
| `FINAL_GMSH_ALGORITHM` | Same categorical choices as `GMSH_ALGORITHM`. |

The reported “internal triangle is coplanar with the cap” error is a geometry/classification check. Changing cap size alone does not address that condition. Its investigation and repair are addressed by the upstream geometry guard.

### Validation, later repairs and tetrahedra

| Control | Increase / decrease or switch effect |
|---|---|
| `MIN_SURFACE_TRIANGLE_QUALITY` | Higher flags more poor triangles; lower accepts more. This is a checking threshold, not an improvement operation. |
| `STOP_ON_VALIDATION_ERRORS` | `True` raises after saving the report; `False` permits later cells but does not make failed geometry valid. |
| `REPAIR_CONTACTS` | Enables the later contact repair; `False` bypasses it. |
| `REPAIR_SEPARATION` | Larger physical movement limit permits stronger separation; smaller is more conservative. Only supported ambiguous edges/fans are handled; genuine multi-grain junction edges remain shared. |
| `REPAIR_INTERFACE_CROSSINGS` | Enables the supported local crossing repair; `False` bypasses it. |
| `CROSSING_MAX_DISPLACEMENT` | Larger permits larger physical corrections; smaller can reject required movement. |
| `CROSSING_NEIGHBOURHOOD_RADIUS` | Larger uses a broader region to estimate separation directions; smaller uses more local normals. It is not a general collision-search distance. |
| `CROSSING_OPENING` | Larger dimensionless value requests wider wedge separation relative to distance from the junction line; smaller is gentler. Excessive opening can violate movement or inversion checks. |
| `RUN_TET_VERIFICATION` | Enables actual tetrahedral generation and checks. Disabling it does not establish readiness. |
| `TET_MESH_SIZE` | Smaller requests finer volume elements at greater cost; larger requests coarser ones, subject to surface constraints. Neither direction guarantees better minimum quality. |
| `TET_OPTIMIZE_NETGEN` | Enables optimization, retaining the baseline for a volume when its minimum quality worsens. |
| `MIN_TET_QUALITY` | Higher makes acceptance stricter; lower admits worse tetrahedra. It does not itself improve elements. |

Crossing repair supports particular interface wedges beside shared junction lines, not arbitrary overlaps or coplanar triangles. Revalidation is required after repairs.

### Element quality

| Control | Effect |
|---|---|
| `THICKNESS_CAP_FRACTION`, `SMOOTH_MIN_WEDGE_DEGREES`, `SMOOTH_ITERATIONS_BY_PAIR` | Differential smoothing: protects necks, keeps grain wedges open, per-interface iteration counts. |
| `SMOOTH_MIN_CORNER_DEGREES` | Smoothing stops before a patch corner at a junction point closes below this. |
| `GMSH_MIN_REMESH_OPENING` | Gmsh remeshes may not create openings sharper than this where the source had none; such patches fall back to the source triangles. |
| `MIN_FACET_OPENING_DEGREES` | Smallest allowed opening between triangles at an edge (grain wedges); default 15. |
| `RELAX_GUARD_ROLLBACK` | Re-smooths the creases the geometry guard leaves. |
| `SURFACE_MIN_ANGLE_DEGREES` | Target smallest triangle angle for the angle repair. |
| `SURFACE_ANGLE_MAX_DEVIATION` | Largest shape change the angle repair may make. |
| `TET_NETGEN_ALL_VOLUMES`, `TET_OPTIMIZE_THRESHOLD` | Tet optimisation effort. |
| `MIN_TET_DIHEDRAL_DEGREES` | Readiness also requires every tet dihedral angle to reach this; `None` reports only. |
| `SMOOTH_TET_DIHEDRALS`, `TET_TARGET_MIN_DIHEDRAL`, `TET_MAX_DIHEDRAL`, `TET_NEIGHBOUR_FLOOR` | Moves tet nodes to raise dihedral angles; tets below the target never get worse, others stay above the floor. |
| `GATE_PASS`, `JUNCTION_POINT_MAX_DEVIATION` | Final smoothing and swaps aimed at the dihedral gate; junction points may move a bounded distance. |
| `SWAP_TETS`, `SMOOTH_AFTER_SWAPS` | Face and edge swaps of poor tets inside grains, then another node-smoothing round. |
| `MOVE_SURFACE_NODES` | Also moves grain-surface nodes along their surface, under the same rule for surface triangles. |
| `APPROVE_TET_MESHING_WITH_POOR_SURFACE` | Tet meshing stops with a warning if any surface triangle angle is below `SURFACE_MIN_ANGLE_DEGREES`, unless this is True. |
| `REQUIRE_READY_TET_MESH` | False (default): tet statistics are reported and the export proceeds; True refuses a mesh below the tet thresholds. |

### Thin-grain freezing

| Control | Effect |
|---|---|
| `FREEZE_THIN_GRAINS` | `False` skips thin-grain freezing. |
| `THIN_CRITERION` | `'inscribed'` freezes grains thin everywhere; `'axial_run'` flags any grain containing a single-voxel run along x, y or z, which freezes nearly every grain. |
| `THIN_THICKNESS_VOXELS` | Higher freezes thicker grains. Inscribed thickness is 1 for grains one or two voxels thick everywhere, 3 for three or four. |
| `EXPLICIT_FROZEN_GRAIN_IDS` | Shuffled IDs frozen in addition. |
| `GMSH_VERBOSE` | Detailed Gmsh messages. |
| Names ending in `_OUTPUT` or `_AUDIT` | Saved-file locations. |

### Final plots

| Control | Effect |
|---|---|
| `SHOW_QUALITY_DISTRIBUTIONS` | Switches the element quality histograms on or off. |
| `QUALITY_WORST_FRACTION` | Fraction of the way from the worst value to the best possible that is shaded and counted in each histogram. |
| `SHOW_TET_VIEW_CLIPPED` | Opens the clipped 3D view of the tetrahedra; the cell waits until its window is closed. |
| `CLIP_CRINKLE`, `CLIP_NORMAL`, `CLIP_ORIGIN`, `CLIP_INVERT` | Crinkle keeps whole tetrahedra cut by the plane; the plane normal, its origin (`None`: the mesh centre) and which side is kept. |
| `SHOW_ALL_GRAINS_CLIPPED` and the `ALL_*` options | The same view and options for all grains at once (`ALL_CLIP_CRINKLE`, `ALL_CLIP_NORMAL`, `ALL_CLIP_ORIGIN`, `ALL_CLIP_INVERT`, ...). |

**Repair scope:** the stages below implement bounded repairs and report unresolved defects. Images alone cannot establish every shared-node contact or triangle intersection.


In [ ]:
from pathlib import Path
import importlib
import numpy as np

# Folder of the bundled sample, relative to the notebook: start Jupyter in this notebook's folder.
DATA_DIR = Path.cwd() / 'tetmesh3d_data' / '25x25x25'

# Full path of the input label array: a 3D .npy array of integer grain IDs.
SAMPLE = DATA_DIR / 'voronoi_50grains.npy'
# SAMPLE = Path(r'C:\path\to\your\labels.npy')  # to use your own file, give its full path

# Folder that receives every result of this notebook (created if missing).
RUN_DIR = DATA_DIR / 'runs' / 'CM04_dev_d3v2p1'
# RUN_DIR = Path(r'C:\path\to\results')

SAMPLE = Path(SAMPLE).expanduser()
RUN_DIR = Path(RUN_DIR).expanduser()
if not SAMPLE.is_file():
    raise FileNotFoundError(f'Sample not found: {SAMPLE}\nSet SAMPLE to the full path of a .npy label array.')
DATA_DIR

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
# Refresh shared helpers as well as entry points in an existing kernel.
for _name in ('facet_angles', 'surface_intersections', 'surface_charts', 'discrete_topology'):
    importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.' + _name))
interface_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.interfaces'))
smooth_interfaces = interface_module.smooth_interfaces
labels_original = np.load(SAMPLE, allow_pickle=False)

GRAIN_ID_SEED = 42
original_grain_ids = np.unique(labels_original)
shuffled_grain_ids = np.random.default_rng(GRAIN_ID_SEED).permutation(original_grain_ids)
grain_id_map = dict(zip(original_grain_ids.tolist(), shuffled_grain_ids.tolist()))
labels = shuffled_grain_ids[np.searchsorted(original_grain_ids, labels_original)]
print(f'{labels.shape}: {len(original_grain_ids)} grains')
import json
SPACING = (1., 1., 1.)


## Optional small-grain removal

`REMOVE_SMALL_GRAINS`: bypass when False. `MIN_GRAIN_VOXELS`: remove candidates strictly smaller than this count **before upscaling**; increasing it considers more grains. Start conservatively at 4. Unsafe candidates remain and are reported.

`PROTECTED_GRAIN_IDS`: shuffled IDs that must survive (they may receive voxels). `PROTECT_RVE_GRAINS`: also protect IDs touching any RVE face. Protect scientifically important islands explicitly. `REMOVAL_MAX_PASSES`: more passes allow revisiting candidates after neighbouring merges. `RECIPIENT_SELECTION`: `shared_area` prefers the largest face-contact area; `largest` prefers the largest neighbouring grain. Ties use grain ID. Spacing from the loading cell weights contact areas.

New cubical boundary-link defects and changes in recipient component count are rejected. Existing input defects still require downstream repair. Whole IDs, including disconnected pieces, are considered together. Surviving IDs are never renumbered. The report includes rejected candidates, mappings and volume changes. This first implementation uses full-grid checks and prioritizes correctness over speed.

In [ ]:
REMOVE_SMALL_GRAINS = True
MIN_GRAIN_VOXELS = 4
PROTECTED_GRAIN_IDS = []
PROTECT_RVE_GRAINS = False
REMOVAL_MAX_PASSES = 10
RECIPIENT_SELECTION = 'shared_area'

_small = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.small_grains'))
labels_reduced, removal_report = _small.remove_small_grains(
    labels, enabled=REMOVE_SMALL_GRAINS, min_voxels=MIN_GRAIN_VOXELS,
    protected_ids=PROTECTED_GRAIN_IDS, protect_rve=PROTECT_RVE_GRAINS,
    max_passes=REMOVAL_MAX_PASSES, selection=RECIPIENT_SELECTION, spacing=SPACING)
retained_grain_ids = np.unique(labels_reduced)
original_to_retained = {int(g): removal_report['original_to_survivor'][int(s)]
                       for g, s in grain_id_map.items()}
REMOVAL_AUDIT = RUN_DIR / 'CM04_dev_small_grains.json'
REMOVAL_AUDIT.parent.mkdir(parents=True, exist_ok=True)
REMOVAL_AUDIT.write_text(json.dumps(dict(report=removal_report,
    original_to_retained=original_to_retained), indent=2))
print('Grains:', len(np.unique(labels)), '->', len(retained_grain_ids))
print('Removed IDs:', removal_report['removed_ids'])
print('Unresolved small IDs:', removal_report['unresolved_small_ids'])
print('Boundary defects:', removal_report['initial_defects'], '->', removal_report['remaining_defects'])


## Iterative UPXO spike removal before upscaling
Uses the existing `StructureCleaner3D._fast_clean_spikes` routine from `upxo.pxtal.twinned_simple_3d.cleaning_3d`. A spike has at most one face-connected neighbor with the same grain ID. Each pass reassigns spikes to neighboring grains while protecting the last voxel of every grain.

Set `CLEAN_SPIKES = False` to bypass cleaning, or adjust `SPIKE_MAX_PASSES`. Cleaning stops when a pass changes no voxels, the pass limit is reached, or a label configuration repeats. Remaining spike candidates can include protected singleton grains. Cleaning changes grain shapes and voxel counts; the original `labels` and input file remain intact. The optional pinch-cleaning step receives `labels_cleaned` before upscaling.


In [ ]:
CLEAN_SPIKES = True
SPIKE_MAX_PASSES = 100

# Always start from loaded/shuffled labels, not the result of a previous run.
labels_cleaned = labels_reduced.copy()
spike_history = []
spike_stop_reason = 'disabled'
if CLEAN_SPIKES:
    import hashlib
    from upxo.pxtal.twinned_simple_3d.cleaning_3d import StructureCleaner3D

    if isinstance(SPIKE_MAX_PASSES, bool) or not isinstance(SPIKE_MAX_PASSES, (int, np.integer)) or SPIKE_MAX_PASSES < 1:
        raise ValueError('SPIKE_MAX_PASSES must be a positive integer')
    if np.any(labels_cleaned < 0):
        raise ValueError('The UPXO spike cleaner requires nonnegative labels.')
    spike_cleaner = StructureCleaner3D(upscale_fallback=False, do_lobe_split=False)
    grain_ids_before_cleaning = np.unique(labels_cleaned)
    seen_states = {hashlib.sha256(labels_cleaned.tobytes()).digest()}
    spike_stop_reason = 'maximum passes reached'
    for pass_number in range(1, SPIKE_MAX_PASSES + 1):
        labels_cleaned, changed = spike_cleaner._fast_clean_spikes(labels_cleaned)
        spike_history.append({'pass': pass_number, 'changed_voxels': int(changed)})
        print(f'Spike pass {pass_number}: {changed} voxels reassigned')
        if changed == 0:
            spike_stop_reason = 'no further changes'
            break
        state = hashlib.sha256(labels_cleaned.tobytes()).digest()
        if state in seen_states:
            spike_stop_reason = 'repeated label configuration'
            break
        seen_states.add(state)
    assert np.array_equal(np.unique(labels_cleaned), grain_ids_before_cleaning), 'Cleaning lost a grain ID'
    remaining_candidates = np.count_nonzero(
        (labels_cleaned > 0) & (spike_cleaner._count_face_same_grain(labels_cleaned)
                               <= spike_cleaner._SPIKE_SAME_COUNT_THRESHOLD))
    print(f'Remaining spike candidates (including protected last voxels): {remaining_candidates}')
print('Spike cleaning stopped:', spike_stop_reason)
print('Net changed voxels:', np.count_nonzero(labels_cleaned != labels))
print('Grains retained:', len(np.unique(labels_cleaned)))


## Optional targeted voxel pinch cleaning
Run after spike removal and before upscaling. `CLEAN_PINCHES` turns the step on or off. The detector searches 2 x 2 x 2 blocks for same-grain voxels connected only through edges/corners locally. It does not apply a majority filter. These are candidates, not proof that the contact is physically wrong; thin necks and pinches introduced later by smoothing are outside this detector's scope.

Each accepted one-voxel edit must remove candidates without introducing new ones, retain every grain ID, and preserve each grain's global face-connected component count. This does not guarantee preservation of tunnels, cavities, or all grain adjacency relationships. Small grains cannot donate voxels. Per-grain net volume change is limited to the greater of one voxel and the specified fraction of its initial count; a zero fraction forbids volume changes.

Controls cap passes, accepted edits, volume change and small-grain protection. Unresolved candidates remain reported. The input `labels_cleaned` remains unchanged; `labels_pinch_cleaned` feeds upscaling. The JSON audit records every voxel reassignment for inspection or replay. Compare on/off results with identical smoothing and Gmsh settings.


In [ ]:
CLEAN_PINCHES = True  # False bypasses this step completely.
PINCH_MAX_PASSES = 100
PINCH_MAX_CHANGES = 200
PINCH_MAX_VOLUME_FRACTION = 0.03
PINCH_PROTECT_GRAINS_UP_TO = 8  # Original voxel count, after spike cleaning.

import json
_pinch_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.pinch_cleaning'))
labels_pinch_cleaned, pinch_report, pinch_changes = _pinch_module.clean_pinch_contacts(
    labels_cleaned, enabled=CLEAN_PINCHES,
    max_passes=PINCH_MAX_PASSES, max_changes=PINCH_MAX_CHANGES,
    max_volume_fraction=PINCH_MAX_VOLUME_FRACTION,
    protect_grains_up_to=PINCH_PROTECT_GRAINS_UP_TO)
print(json.dumps(pinch_report, indent=2))
PINCH_AUDIT = RUN_DIR / 'CM04_dev_pinch_cleaning.json'
PINCH_AUDIT.parent.mkdir(parents=True, exist_ok=True)
PINCH_AUDIT.write_text(json.dumps({'report': pinch_report, 'changes': pinch_changes}, indent=2))
print('Pinch-cleaning audit:', PINCH_AUDIT)


### Rigorous voxel boundary topology cleaning

This checks whether every grain has a single, non-branching boundary link at each lattice vertex, including RVE faces, edges and corners. It detects point contacts, alternating edge contacts and disconnected complementary regions. Single-voxel edits and bounded two-voxel edits reduce the defect count; every grain ID is retained. This is targeted topology cleaning, not a majority filter.

Higher pass/change/pair-trial budgets allow more search. A higher volume fraction permits larger **net** per-grain count changes; zero still permits count-preserving swaps. Raising the small-grain protection threshold protects more donors. `TOPOLOGY_PRESERVE_COMPONENTS=False` permits changes to connected-component counts, including removal of legitimate contacts; set True to restrict them. `REQUIRE_CLEAN_TOPOLOGY` stops on remaining defects. The audit records all edits and unresolved vertices. All switches are user-controlled.

A positive volume fraction allows at least one voxel of net count change per grain; this discrete minimum can exceed that fraction for very small grains. Zero allows only count-preserving swaps. Set `TOPOLOGY_MAX_PAIR_TRIALS=0` to disable two-voxel lookahead.


In [ ]:
# Exact cubical boundary-link cleaning, including RVE boundary vertices.
CLEAN_VOXEL_TOPOLOGY = True
TOPOLOGY_MAX_PASSES = 20
TOPOLOGY_MAX_CHANGES = 5000
TOPOLOGY_MAX_VOLUME_FRACTION = 0.20
TOPOLOGY_PROTECT_GRAINS_UP_TO = 0
TOPOLOGY_PRESERVE_COMPONENTS = False
TOPOLOGY_MAX_PAIR_TRIALS = 20000
REQUIRE_CLEAN_TOPOLOGY = True

_topology = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.voxel_topology'))
labels_meshing, topology_report, topology_changes = _topology.clean_voxel_topology(
    labels_pinch_cleaned, enabled=CLEAN_VOXEL_TOPOLOGY,
    max_passes=TOPOLOGY_MAX_PASSES, max_changes=TOPOLOGY_MAX_CHANGES,
    max_volume_fraction=TOPOLOGY_MAX_VOLUME_FRACTION,
    protect_grains_up_to=TOPOLOGY_PROTECT_GRAINS_UP_TO,
    preserve_components=TOPOLOGY_PRESERVE_COMPONENTS,
    max_pair_trials=TOPOLOGY_MAX_PAIR_TRIALS)
TOPOLOGY_AUDIT = RUN_DIR / 'CM04_dev_voxel_topology.json'
TOPOLOGY_AUDIT.parent.mkdir(parents=True, exist_ok=True)
TOPOLOGY_AUDIT.write_text(json.dumps({'report': topology_report, 'changes': topology_changes}, indent=2))
print({k:v for k,v in topology_report.items() if k not in ('unresolved', 'grain_voxel_changes')})
assert np.array_equal(np.unique(labels_meshing), retained_grain_ids)
if REQUIRE_CLEAN_TOPOLOGY and topology_report['remaining_defects']:
    raise RuntimeError('Unresolved voxel boundary defects: inspect TOPOLOGY_AUDIT before continuing.')


## Freeze thin grains before upscaling

Grains that are thin everywhere collapse under smoothing and leave sliver tetrahedra, so their whole voxel geometry is frozen through every later stage.

`THIN_CRITERION = 'inscribed'` measures each grain's inscribed thickness, 2*r - 1 voxels, where r is the largest distance from a grain voxel to the nearest voxel outside it (the RVE exterior counts as outside). A grain is frozen when this is at most `THIN_THICKNESS_VOXELS`: 1 freezes grains at most two voxels thick everywhere, 3 at most four. A large grain with a thin tip or staircase corner is not frozen. `'axial_run'` is the alternative rule, which flags any grain containing a single-voxel run along x, y or z and so freezes nearly every grain.

Frozen grains keep their voxel facets (Gmsh may retriangulate a facet but not round its corners); every later stage is checked against them and stops on a conflict. `EXPLICIT_FROZEN_GRAIN_IDS` adds shuffled IDs by hand. The audit lists every frozen ID with its reason and the inscribed thickness of every grain.

In [ ]:
FREEZE_THIN_GRAINS = False  # True keeps thin grains as voxel facets (more voxel-like boundaries)
AUTO_FREEZE_THIN_GRAINS = True
THIN_CRITERION = 'inscribed'   # or 'axial_run' (CM05)
THIN_THICKNESS_VOXELS = 1
EXPLICIT_FROZEN_GRAIN_IDS = []

_freeze = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.thin_freeze'))
FROZEN_GRAIN_IDS, freezing_report = _freeze.detect_thin_grains(
    labels_meshing, enabled=FREEZE_THIN_GRAINS, automatic=AUTO_FREEZE_THIN_GRAINS,
    thickness=THIN_THICKNESS_VOXELS, explicit_ids=EXPLICIT_FROZEN_GRAIN_IDS,
    criterion=THIN_CRITERION)
frozen_geometry = _freeze.FrozenGeometry(labels_meshing, FROZEN_GRAIN_IDS, SPACING)
FREEZING_AUDIT = RUN_DIR / 'CM04_dev_frozen_grains.json'
FREEZING_AUDIT.write_text(json.dumps(freezing_report, indent=2, default=float))
frozen_voxels = int(np.isin(labels_meshing, FROZEN_GRAIN_IDS).sum())
print(f'Frozen {len(FROZEN_GRAIN_IDS)} / {len(np.unique(labels_meshing))} grains '
      f'({frozen_voxels / labels_meshing.size:.1%} of the volume)')
print('Frozen IDs:', FROZEN_GRAIN_IDS)

## Optional resolution increase before smoothing
Set `UPSCALE_FACTOR` to an integer: `1` leaves the data unchanged, `2` doubles the resolution along each axis, and `3` triples it. Each original voxel is subdivided into identically labelled voxels using nearest-neighbor replication; no new grain IDs are introduced. Spacing decreases by the same factor, preserving physical RVE dimensions and grain shapes before smoothing.

For the 25 x 25 x 25 sample, factor 2 produces 50 x 50 x 50 voxels; factor 3 produces 75 x 75 x 75. This notebook limits refinement to a maximum factor of 3. Voxel memory grows as factor cubed, and interface triangle count as factor squared. At higher resolution, the same Laplacian iteration count generally smooths over a smaller physical distance; increase `LAPLACE_ITERATIONS` if needed. Always refine from `labels_meshing`, so rerunning this cell does not compound the upscaling.


In [ ]:
UPSCALE_FACTOR = 1  # 1: unchanged; 2 or 3: subdivide along all three axes.
if UPSCALE_FACTOR not in (1, 2, 3):
    raise ValueError("CM04 supports upscale factors 1, 2, or 3 (maximum).")
# SPACING is set in the loading cell; original value = (1.0, 1.0, 1.0)  # Original voxel spacing, before refinement.

# Refresh definitions if the notebook kernel predates this module update.
interface_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.interfaces'))
smooth_interfaces = interface_module.smooth_interfaces
labels_upscaled, effective_spacing = interface_module.upscale_labels(
    labels_meshing, factor=UPSCALE_FACTOR, spacing=SPACING)
print(f'Voxel shape: {labels_meshing.shape} -> {labels_upscaled.shape}')
print(f'Effective voxel spacing: {effective_spacing}')
print(f'Physical RVE dimensions: {np.array(labels_upscaled.shape) * effective_spacing}')


## Extract and smooth interfaces only
Change the iteration count and relaxation to control smoothing. Shared interfaces are extracted once per adjacent grain pair. This stage works on surface triangles only; volume meshing comes later.

Junction edges are identified from their incident grain interfaces, avoiding false branches caused by surface triangulation diagonals. Genuine junction points (four or more grains), branches and endpoints remain fixed. Movable degree-two junction nodes have independent controls:

- `JUNCTION_ITERATIONS`: line-smoothing steps; 0 disables line motion.
- `JUNCTION_RELAXATION`: smoothing rate; 0.25 is a damped starting value.
- `JUNCTION_MAX_DISPLACEMENT`: maximum total movement from the extracted coordinates, in physical units. The default uses 0.75 original voxel widths, independently of upscaling.

Surface smoothing and line smoothing advance together until their respective iteration limits. Triangle-collapse checks can restrict individual moves; frozen endpoints and branches can retain sharp turns. Increase surface iterations if surrounding surfaces constrain line motion. Turning-angle diagnostics refer only to movable degree-two nodes (0 degrees is straight).


**Differential smoothing:** `THICKNESS_CAP_FRACTION` limits how far each node moves from its voxel position to that fraction of half the local grain thickness, so thin necks and plates keep their thickness while thick regions smooth fully. `SMOOTH_MIN_WEDGE_DEGREES` stops nodes for the rest of the smoothing when a move would close a grain wedge or fold below that opening. `SMOOTH_ITERATIONS_BY_PAIR` sets the number of Laplacian iterations for chosen grain-pair interfaces; `SMOOTH_RAMP_RINGS` blends the counts over neighbouring nodes. `None` switches each off.

In [ ]:
LAPLACE_ITERATIONS = 50
LAPLACE_RELAXATION = 0.75

# Independent controls for actual junction-line edges.
JUNCTION_ITERATIONS = 10
JUNCTION_RELAXATION = 0.35  # Damped Laplacian step; 0 < value <= 1.
JUNCTION_MAX_DISPLACEMENT = 0.75 * min(SPACING)  # Physical units, from extracted geometry.

# Differential smoothing (None switches each off).
THICKNESS_CAP_FRACTION = 0.5  # node moves at most this fraction of the local half-thickness
SMOOTH_MIN_WEDGE_DEGREES = 30.0  # nodes stop before a grain wedge or fold closes below this
SMOOTH_MIN_CORNER_DEGREES = 20.0  # nodes stop before a patch corner at a junction point closes below this
SMOOTH_ITERATIONS_BY_PAIR = None  # e.g. {(12, 40): 10}: Laplacian iterations for that interface
SMOOTH_RAMP_RINGS = 2  # rings over which per-pair iteration counts are blended

# Refresh the function even when only this cell is rerun after a source edit.
interface_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.interfaces'))
smooth_interfaces = interface_module.smooth_interfaces
interfaces = smooth_interfaces(labels_upscaled, spacing=effective_spacing,
                               iterations=LAPLACE_ITERATIONS,
                               relaxation=LAPLACE_RELAXATION,
                               junction_iterations=JUNCTION_ITERATIONS,
                               junction_relaxation=JUNCTION_RELAXATION,
                               junction_max_displacement=JUNCTION_MAX_DISPLACEMENT,
                               frozen_grain_ids=FROZEN_GRAIN_IDS,
                               iterations_by_pair=SMOOTH_ITERATIONS_BY_PAIR, ramp_rings=SMOOTH_RAMP_RINGS,
                               thickness_cap_fraction=THICKNESS_CAP_FRACTION,
                               minimum_wedge_angle=SMOOTH_MIN_WEDGE_DEGREES,
                               minimum_corner_angle=SMOOTH_MIN_CORNER_DEGREES)
frozen_geometry.check(interfaces)
assert np.array_equal(interfaces.points[interfaces.node_kind == 3],
                      interfaces.original_points[interfaces.node_kind == 3])
assert np.array_equal(interfaces.points[interfaces.fixed_axes],
                      interfaces.original_points[interfaces.fixed_axes])
print(f'{len(interfaces.points):,} surface nodes; {len(interfaces.triangles):,} interface triangles')
print(f'Accepted Laplacian iterations: {interfaces.accepted_iterations}')
print('Frozen junction points:', np.count_nonzero(interfaces.node_kind == 3))
import json
print(json.dumps(interfaces.junction_metrics(), indent=2))
print('Differential smoothing:', json.dumps(interfaces.smoothing_report, indent=2))

# Stable input for rerunning the optional point-contact separation cell.
interfaces_smoothed = interfaces


### Check and stabilize smoothed geometry before Gmsh

Smoothing can introduce intersections even when voxel topology is valid. The guard checks triangle intersections, source-relative collapse/reversal and approach to RVE planes. Affected nodes and one neighbour ring are pulled back toward extracted voxel coordinates. Frozen junctions and existing RVE coordinates stay fixed. Local staircase detail may return where required; no triangles or grains are deleted.

Larger clearance keeps originally interior nodes farther from box planes (capped by their original distance). More passes permit more repair attempts. `GEOMETRY_ROLLBACK_RELAXATION` is the retained fraction of smoothing displacement per retry: **smaller values restore voxel detail more strongly**, larger values more gradually. A finite maximum correction bounds movement in physical units; None permits complete rollback. Disabled mode reports unverified geometry. A failed enabled repair raises.

The Gmsh cell additionally checks intersections/cap tangencies. Its bounded retries restore the source triangulation on affected charts, limiting geometric approximation there. These checks do not replace later closed-volume and tetrahedral-quality validation.


**Near-contact opening:** `MIN_FACET_OPENING_DEGREES` rejects almost closed wedges between triangles sharing an edge, including triple lines. Increasing it restores more local voxel detail and limits near-singular sheets; decreasing it permits sharper wedges. `0` disables this check. The default is 15 degrees, which rejects the knife-edge grain wedges (3-9 degrees) that produced the remaining poor tetrahedra with a 1 degree limit. Any value in [0, 180) is accepted; Gmsh can reject angles below its 0.1 degree overlap tolerance even when there is no mathematical intersection. This control is checked before and after Gmsh and during quality repair. Frozen junction points and RVE-plane coordinates remain fixed.


In [ ]:
# Restore local voxel detail where smoothing creates invalid geometry.
GUARD_SMOOTHED_GEOMETRY = True
MIN_FACET_OPENING_DEGREES = 15.0  # degrees, user input; 0 disables. CM04 uses 1.0
GEOMETRY_BOUNDARY_CLEARANCE = 0.05 * float(np.min(effective_spacing))
GEOMETRY_MAX_PASSES = 100
GEOMETRY_ROLLBACK_RELAXATION = 0.5
GEOMETRY_MAX_CORRECTION = None  # None permits return to original voxel geometry.

_guard = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.geometry_guard'))
interfaces, geometry_report = frozen_geometry.run(_guard.stabilize_interfaces,
    interfaces_smoothed, np.array(labels_upscaled.shape) * effective_spacing,
    enabled=GUARD_SMOOTHED_GEOMETRY, boundary_clearance=GEOMETRY_BOUNDARY_CLEARANCE,
    max_passes=GEOMETRY_MAX_PASSES, relaxation=GEOMETRY_ROLLBACK_RELAXATION,
    max_correction=GEOMETRY_MAX_CORRECTION,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
GEOMETRY_AUDIT = RUN_DIR / 'CM04_dev_geometry_guard.json'
GEOMETRY_AUDIT.write_text(json.dumps(geometry_report, indent=2))
print(json.dumps(geometry_report, indent=2))
interfaces_before_unpinch = interfaces


### Re-smooth guard rollback regions (optional)

The guard pulls defective neighbourhoods back toward voxel coordinates, which leaves creases at the edge of each rollback region. `RELAX_GUARD_ROLLBACK` applies `GUARD_RELAX_ITERATIONS` further Laplacian steps to rolled-back nodes and `GUARD_RELAX_MARGIN_RINGS` rings of neighbours only. A node keeps its move only while every triangle it touches passes the guard's checks (intersections, `MIN_FACET_OPENING_DEGREES`, orientation, RVE clearance). Fixed junction points, RVE-plane coordinates, frozen grains and `JUNCTION_MAX_DISPLACEMENT` are kept. Fold counts (neighbouring triangles of one interface meeting at an opening below 150 and 120 degrees) are printed before and after.

In [ ]:
RELAX_GUARD_ROLLBACK = True
GUARD_RELAX_ITERATIONS = 3
GUARD_RELAX_MARGIN_RINGS = 2

_relax = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.guard_relaxation'))
interfaces, guard_relaxation_report = frozen_geometry.run(_relax.relax_guarded_interfaces,
    interfaces, interfaces_smoothed, np.array(labels_upscaled.shape) * effective_spacing,
    enabled=RELAX_GUARD_ROLLBACK, iterations=GUARD_RELAX_ITERATIONS,
    margin_rings=GUARD_RELAX_MARGIN_RINGS,
    minimum_facet_angle=max(MIN_FACET_OPENING_DEGREES, SMOOTH_MIN_WEDGE_DEGREES or 0.),  # keep the smoothing wedge limit
    boundary_clearance=GEOMETRY_BOUNDARY_CLEARANCE,
    junction_max_displacement=JUNCTION_MAX_DISPLACEMENT, exclude_grain_ids=FROZEN_GRAIN_IDS)
GUARD_RELAXATION_AUDIT = RUN_DIR / 'CM04_dev_guard_relaxation.json'
GUARD_RELAXATION_AUDIT.write_text(json.dumps(guard_relaxation_report, indent=2))
print(json.dumps({k: v for k, v in guard_relaxation_report.items() if k != 'history'}, indent=2))
interfaces_before_unpinch = interfaces

## Separate point contacts and press down their tips before Gmsh
This optional step intentionally removes even legitimate point contacts when the incident triangles form disconnected surface fans. Each fan receives its own node and its tip is relaxed inward with a smoothly decaying influence over the surrounding surface. Topology can change, and grain volumes are not preserved.

`SEPARATE_PINCHES` turns this on/off. Radius, maximum movement, and minimum opening are in physical units; the defaults scale with the original voxel spacing, independently of upscaling. Only the nodes at selected contacts are released from their former common-node/frozen constraints. Other frozen junction points remain fixed, and RVE coordinates stay on their original planes.

The step preserves edge incidence, guards against triangle collapse and per-step reversal, and rolls back the batch if any duplicated tips cannot be opened sufficiently. Connected surface fans and shared junction edges are not cut; separating those would require reconstructing interface patches. It does not certify the absence of global intersections or produce a tetrahedral mesh.

Rerun the smoothing cell once to populate `interfaces_before_unpinch`. After that, this cell can be rerun with different settings without compounding its previous changes. The Gmsh cell below uses the resulting `interfaces`.


In [ ]:
SEPARATE_PINCHES = True
PINCH_PRESS_RADIUS = 1.5 * min(SPACING)
PINCH_PRESS_MAX_DISPLACEMENT = 0.5 * min(SPACING)
PINCH_PRESS_ITERATIONS = 30
PINCH_PRESS_RELAXATION = 0.25
PINCH_MIN_OPENING = 0.02 * min(SPACING)

import json
_unpinch_module = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.unpinch'))
interfaces, unpinch_report, unpinch_audit = frozen_geometry.run(_unpinch_module.separate_pinch_fans,
    interfaces_before_unpinch, enabled=SEPARATE_PINCHES,
    radius=PINCH_PRESS_RADIUS, max_displacement=PINCH_PRESS_MAX_DISPLACEMENT,
    iterations=PINCH_PRESS_ITERATIONS, relaxation=PINCH_PRESS_RELAXATION,
    min_separation=PINCH_MIN_OPENING)
print(json.dumps(unpinch_report, indent=2))
UNPINCH_AUDIT = RUN_DIR / 'CM04_dev_unpinch.json'
UNPINCH_AUDIT.parent.mkdir(parents=True, exist_ok=True)
UNPINCH_AUDIT.write_text(json.dumps({'report': unpinch_report, 'contacts': unpinch_audit}, indent=2))
print('Point-contact repair audit:', UNPINCH_AUDIT)


## Gmsh remeshing with connected interface boundaries
Gmsh builds shared discrete curves from the labelled surface topology, parametrizes each surface, and regenerates its 2D triangles. Existing curve nodes and edges are retained: neighboring interfaces therefore share the same boundary discretization, and traces on RVE faces stay on those faces at their smoothed positions. Frozen junction points also remain fixed. No volume mesh or exterior caps are generated.

`GMSH_SIZE` is the target interior triangle size in physical units; fixed boundary segments retain their existing lengths. Parametrization can split a grain-pair interface into several charts. The step checks every chart boundary and preserves all grain-pair labels; failures raise an error instead of silently skipping an interface. The original `interfaces` object is unchanged.

Uses Gmsh's [discrete geometry and topology APIs](https://gmsh.info/doc/texinfo/gmsh.html#Namespace-gmsh_002fmodel_002fmesh), with `Mesh.MeshOnlyEmpty` to retain the curve mesh. Requires `gmsh` and `pyvista` in the selected kernel (verified with Gmsh 4.14.1). The VTP export retains both grain IDs for every triangle.


Change `GMSH_ALGORITHM` to compare triangular surface meshers: 1 MeshAdapt, 2 Automatic, 5 Delaunay, 6 Frontal-Delaunay (the previous default). Keep geometry and `GMSH_SIZE` fixed when comparing. Gmsh can fall back internally to MeshAdapt; use `GMSH_VERBOSE = True` to inspect its messages. The report records the requested algorithm, not a guarantee of which fallback ran. Shared-curve and RVE constraints remain checked for every option. Changing the algorithm does not repair a distorted underlying parametrization.


**Gmsh patch size:** `GMSH_CHART_TRIANGLES` bounds the number of source triangles in each parametrization patch. Decrease it for smaller, more constrained patches; increase it for more freedom to remesh across the surface. Extra patch boundaries remain shared, so they do not create gaps. `None` uses automatic topology partitioning. Intersection checks remain necessary regardless of this setting.


In [ ]:
# Regenerate all grain-interface surface triangles in Gmsh.
import importlib
import json
import pyvista as pv  # file export only

GMSH_SIZE = 0.75 * min(SPACING)  # Physical target size; boundary curves stay fixed.
GMSH_ALGORITHM = 6  # 1: MeshAdapt; 2: Automatic; 5: Delaunay; 6: Frontal-Delaunay
GMSH_CHECK_INTERSECTIONS = True
GMSH_INTERSECTION_RETRIES = 4
GMSH_CHART_TRIANGLES = 128  # Maximum source triangles per parametrization patch
GMSH_VERBOSE = False
GMSH_MIN_REMESH_OPENING = 30.0  # sharper openings created by a remesh fall back to source; None disables

_gmsh_remesher = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p1.gmsh_interfaces'))
gmsh_result = frozen_geometry.run(_gmsh_remesher.remesh_interfaces_gmsh,
    interfaces, mesh_size=GMSH_SIZE, verbose=GMSH_VERBOSE,
    algorithm=GMSH_ALGORITHM, check_intersections=GMSH_CHECK_INTERSECTIONS,
    intersection_retries=GMSH_INTERSECTION_RETRIES,
    max_chart_triangles=GMSH_CHART_TRIANGLES,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES, minimum_remesh_opening=GMSH_MIN_REMESH_OPENING,
    rve_dimensions=np.array(labels_upscaled.shape) * effective_spacing)
print(json.dumps(gmsh_result.report, indent=2))

cells = np.column_stack((np.full(len(gmsh_result.triangles), 3), gmsh_result.triangles)).ravel()
gmsh_surfaces = pv.PolyData(gmsh_result.points, cells)
gmsh_surfaces.cell_data['grain_a'] = gmsh_result.grain_pairs[:, 0]
gmsh_surfaces.cell_data['grain_b'] = gmsh_result.grain_pairs[:, 1]
GMSH_OUTPUT = RUN_DIR / 'CM04_dev_gmsh.vtp'
GMSH_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
gmsh_surfaces.save(GMSH_OUTPUT)
GMSH_OUTPUT.with_suffix('.json').write_text(json.dumps(gmsh_result.report, indent=2))
print('Saved:', GMSH_OUTPUT)


## Close all six RVE faces
Build grain-labelled planar caps from the current Gmsh interface traces and the RVE perimeter. Boundary segments reuse the existing interface node IDs without moving them or subdividing their edges. Caps are triangulated by Gmsh and joined into one surface complex.

This cell checks complete face coverage, a closed exterior shell, per-grain edge closure, positive enclosed signed grain volumes and total RVE volume. Existing nonmanifold grain edges are reported separately; capping does not repair those or certify absence of self-intersections. No tetrahedra are generated here.

Run the preceding Gmsh cell first. `CAP_MESH_SIZE` controls cap-interior resolution; boundary segments remain fixed. The complete closed surface complex, the exterior caps alone, and a validation report are exported. On cap triangles `grain_a == grain_b`; `is_rve_cap` distinguishes them from internal interfaces. Face IDs 0..5 mean x-min, x-max, y-min, y-max, z-min, z-max.

Box perimeter edges are subdivided using `RVE_EDGE_SIZE`; adjacent RVE faces share these nodes. Rerun this closure cell, then the final joint remeshing cell after changing the size.


In [ ]:
import importlib
import json
import pyvista as pv

CAP_MESH_SIZE = GMSH_SIZE
RVE_EDGE_SIZE = CAP_MESH_SIZE  # Maximum new box-edge segment length

_rve_caps = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.rve_caps'))
closed_rve = frozen_geometry.run(_rve_caps.close_rve_faces,
    gmsh_result, labels_upscaled, spacing=effective_spacing,
    mesh_size=CAP_MESH_SIZE, edge_size=RVE_EDGE_SIZE)
print(json.dumps({k: v for k, v in closed_rve.report.items()
                  if k != 'enclosed_grain_volumes'}, indent=2))

closed_cells = np.column_stack((np.full(len(closed_rve.triangles), 3),
                                closed_rve.triangles)).ravel()
closed_rve_surface = pv.PolyData(closed_rve.points, closed_cells)
closed_rve_surface.cell_data['grain_a'] = closed_rve.grain_pairs[:, 0]
closed_rve_surface.cell_data['grain_b'] = closed_rve.grain_pairs[:, 1]
closed_rve_surface.cell_data['is_rve_cap'] = closed_rve.exterior.astype(np.uint8)
closed_rve_surface.cell_data['rve_face'] = closed_rve.rve_face
rve_caps_surface = closed_rve_surface.extract_cells(closed_rve.exterior).extract_surface()

CLOSED_RVE_OUTPUT = RUN_DIR / 'CM04_dev_closed_RVE.vtp'
CLOSED_RVE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
closed_rve_surface.save(CLOSED_RVE_OUTPUT)
rve_caps_surface.save(CLOSED_RVE_OUTPUT.with_name('CM04_dev_RVE_caps.vtp'))
CLOSED_RVE_OUTPUT.with_suffix('.json').write_text(json.dumps(closed_rve.report, indent=2))
print('Saved closed surface complex:', CLOSED_RVE_OUTPUT)


### Joint Gmsh remeshing: RVE caps and internal interfaces
Run after the RVE closure cell. All labelled surface patches are regenerated together. Shared curve nodes/edges stay fixed; cap interior nodes are remeshed on their RVE planes. This is surface meshing only. The report checks closure and lists any nonmanifold grain edges that still require repair before tetrahedral meshing.


### Remove surface slivers before the joint remesh

`PRE_JOINT_QUALITY_REPAIR` enables guarded diagonal flips before Gmsh sees the closed surface. This reduces nearly collinear input triangles that can cause parametrization failures. `PRE_JOINT_QUALITY_TARGET` (0–1) targets more triangles when increased; `PRE_JOINT_QUALITY_MAX_PASSES` allows more attempts at additional cost. Node coordinates and grain/RVE patch boundaries remain fixed; local curved shape and grain volumes can change. The later quality-repair cell performs the same checks on the final remeshed result.


In [ ]:
PRE_JOINT_QUALITY_REPAIR = True
PRE_JOINT_QUALITY_TARGET = 0.05
PRE_JOINT_QUALITY_MAX_PASSES = 10

_quality = importlib.reload(importlib.import_module('upxo.meshing.gbconformant.d3v2p0.surface_quality'))
closed_rve = frozen_geometry.run(_quality.improve_surface_quality,
    closed_rve, enabled=PRE_JOINT_QUALITY_REPAIR,
    minimum_quality=PRE_JOINT_QUALITY_TARGET, max_passes=PRE_JOINT_QUALITY_MAX_PASSES,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
print(json.dumps(closed_rve.report['surface_quality_repair'], indent=2))
CLOSED_RVE_OUTPUT.with_name('CM04_dev_pre_joint_quality.json').write_text(
    json.dumps(closed_rve.report['surface_quality_repair'], indent=2))


In [ ]:
# Remesh all caps and internal interfaces in one Gmsh model.
FINAL_GMSH_SIZE = GMSH_SIZE
FINAL_GMSH_ALGORITHM = GMSH_ALGORITHM  # 1, 2, 5, or 6

from importlib import import_module, reload
reload(import_module('upxo.meshing.gbconformant.d3v2p1.gmsh_interfaces'))
_joint = reload(import_module('upxo.meshing.gbconformant.d3v2p1.gmsh_closed'))
remeshed_rve = frozen_geometry.run(_joint.remesh_closed_rve_gmsh,
    closed_rve, mesh_size=FINAL_GMSH_SIZE, algorithm=FINAL_GMSH_ALGORITHM,
    check_intersections=GMSH_CHECK_INTERSECTIONS, intersection_retries=GMSH_INTERSECTION_RETRIES,
    max_chart_triangles=GMSH_CHART_TRIANGLES,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES, minimum_remesh_opening=GMSH_MIN_REMESH_OPENING)
print({k: v for k, v in remeshed_rve.report.items()
       if k != 'enclosed_grain_volumes'})

remeshed_rve_surface = pv.PolyData(
    remeshed_rve.points,
    np.column_stack((np.full(len(remeshed_rve.triangles), 3),
                     remeshed_rve.triangles)).ravel())
for key, values in dict(grain_a=remeshed_rve.grain_pairs[:, 0],
                        grain_b=remeshed_rve.grain_pairs[:, 1],
                        is_rve_cap=remeshed_rve.exterior.astype(np.uint8),
                        rve_face=remeshed_rve.rve_face).items():
    remeshed_rve_surface.cell_data[key] = values
remeshed_rve_caps = remeshed_rve_surface.extract_cells(
    remeshed_rve.exterior).extract_surface()
FINAL_RVE_OUTPUT = CLOSED_RVE_OUTPUT.with_name('CM04_dev_closed_RVE_remeshed.vtp')
remeshed_rve_surface.save(FINAL_RVE_OUTPUT)
import json
FINAL_RVE_OUTPUT.with_suffix('.json').write_text(
    json.dumps(remeshed_rve.report, indent=2))


### Pre-tetrahedral-meshing surface validation
Run after the final joint surface remeshing cell. This read-only check reports per-grain closure, edge/vertex manifoldness, orientation, ownership, duplicate triangles/nodes, triangle quality, RVE planes/areas and signed volumes. Shared triple lines are checked per grain. `BLOCKED` means defects were found. A preliminary pass is **not** a guarantee of successful tetrahedral meshing: triangle intersections, clearance and shell containment remain unchecked. No tetrahedra are generated.


In [ ]:
# Validate the final surface complex before attempting Gmsh volume meshing.
MIN_SURFACE_TRIANGLE_QUALITY = 0.05
STOP_ON_VALIDATION_ERRORS = False  # True raises after saving the report.

from importlib import import_module, reload
import json
_validation = reload(import_module('upxo.meshing.gbconformant.d3v2p1.tet_validation'))
tet_validation = _validation.validate_tet_surfaces(
    remeshed_rve, expected_grains=np.unique(labels_upscaled),
    min_triangle_quality=MIN_SURFACE_TRIANGLE_QUALITY, check_intersections=True, minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
print('Pre-tet validation:', tet_validation['status'])
print('Triangle quality:', tet_validation.get('triangle_quality'))
for category in ('blockers', 'warnings', 'unchecked'):
    print('\n' + category.upper())
    for message in tet_validation[category]:
        print(' -', message)
print('\nPer-grain defects:')
for grain, checks in tet_validation['per_grain'].items():
    if any(checks[k] for k in ('open_edges', 'nonmanifold_edges',
                               'inconsistent_oriented_edges', 'nonmanifold_vertices')):
        print(grain, checks)
VALIDATION_OUTPUT = FINAL_RVE_OUTPUT.with_name('CM04_dev_pre_tet_validation.json')
VALIDATION_OUTPUT.write_text(json.dumps(tet_validation, indent=2))
print('Saved:', VALIDATION_OUTPUT)

if STOP_ON_VALIDATION_ERRORS and tet_validation['blockers']:
    raise RuntimeError('Surface validation blocked: inspect tet_validation and its JSON report.')


### Repair contacts, revalidate, and verify tetrahedral meshing
These cells use the final `remeshed_rve` without altering it. The repair separates disconnected sheets at point/edge contacts and moves only those contact nodes by a bounded amount. True shared junctions remain connected and box-plane coordinates remain fixed. This deliberately changes local grain morphology. Unsupported defects stop the subsequent volume-meshing cell.

The final cell generates a real Gmsh tetrahedral mesh, including disconnected pieces with their original grain IDs. It checks positive element volumes, every grain's surface conformity and volume, and minimum tetrahedron quality. `TET_MESH_VERIFIED` applies to the generated mesh and selected quality threshold; it is not a universal guarantee for different settings or future datasets.

The intersection-repair cell separates crossing interface wedges while freezing their shared junctions. Final validation and the tetrahedral mesher both test geometric triangle intersections, including crossings between triangles that share a node. These are tolerance-based checks; minimum-clearance certification remains separate.


In [ ]:
from importlib import import_module, reload
import json

REPAIR_CONTACTS = True
REPAIR_SEPARATION = 0.15 * float(np.min(SPACING))  # Maximum movement in physical units

_repair = reload(import_module('upxo.meshing.gbconformant.d3v2p0.tet_repair'))
repaired_rve = (frozen_geometry.run(_repair.repair_tet_contacts, remeshed_rve, separation=REPAIR_SEPARATION)
                if REPAIR_CONTACTS else remeshed_rve)
print('Contact repair:', repaired_rve.report.get('contact_repair', 'disabled'))
repaired_rve_surface = pv.PolyData(
    repaired_rve.points,
    np.column_stack((np.full(len(repaired_rve.triangles), 3), repaired_rve.triangles)).ravel())
repaired_rve_surface.cell_data['grain_a'] = repaired_rve.grain_pairs[:, 0]
repaired_rve_surface.cell_data['grain_b'] = repaired_rve.grain_pairs[:, 1]
repaired_rve_surface.cell_data['is_rve_cap'] = repaired_rve.exterior.astype(np.uint8)
repaired_rve_surface.cell_data['rve_face'] = repaired_rve.rve_face
REPAIRED_OUTPUT = FINAL_RVE_OUTPUT.with_name('CM04_dev_repaired_RVE.vtp')
repaired_rve_surface.save(REPAIRED_OUTPUT)


In [ ]:
# Remove geometric crossings beside junction lines before the final checks.
REPAIR_INTERFACE_CROSSINGS = True
CROSSING_MAX_DISPLACEMENT = 0.35 * float(np.min(SPACING))
CROSSING_NEIGHBOURHOOD_RADIUS = 1.0 * float(np.min(SPACING))
CROSSING_OPENING = 0.10  # Separation relative to distance from the junction line

_crossings = reload(import_module('upxo.meshing.gbconformant.d3v2p0.intersection_repair'))
if REPAIR_INTERFACE_CROSSINGS:
    repaired_rve = frozen_geometry.run(_crossings.repair_interface_crossings,
        repaired_rve, max_displacement=CROSSING_MAX_DISPLACEMENT,
        radius=CROSSING_NEIGHBOURHOOD_RADIUS, opening=CROSSING_OPENING)
    print('Intersection repair:', repaired_rve.report['intersection_repair'])
    repaired_rve_surface.points = repaired_rve.points
    repaired_rve_surface.save(REPAIRED_OUTPUT)
else:
    print('Intersection repair disabled; the following checks still detect crossings.')


### Improve poor surface triangles

`REPAIR_SURFACE_QUALITY` turns guarded diagonal flips on/off. Node coordinates, grain junction edges and RVE patch boundaries stay fixed. A flip must improve the minimum quality of its two triangles and pass intersection checks against the full complex. Flips on curved patches may change local surface shape and individual grain volumes; the shared complex remains connected.

`SURFACE_QUALITY_TARGET` uses the same 0–1 triangle metric as validation. Increase it to target more triangles; decrease it to accept more elongated triangles. `SURFACE_QUALITY_MAX_PASSES` allows more local attempts when increased, at additional cost. Some constrained triangles cannot be improved by flips; their remaining count is reported. This does not guarantee the tetrahedral quality target. The following full validation remains required.


In [ ]:
REPAIR_SURFACE_QUALITY = True
SURFACE_QUALITY_TARGET = MIN_SURFACE_TRIANGLE_QUALITY
SURFACE_QUALITY_MAX_PASSES = 10

_quality = reload(import_module('upxo.meshing.gbconformant.d3v2p0.surface_quality'))
repaired_rve = frozen_geometry.run(_quality.improve_surface_quality,
    repaired_rve, enabled=REPAIR_SURFACE_QUALITY,
    minimum_quality=SURFACE_QUALITY_TARGET, max_passes=SURFACE_QUALITY_MAX_PASSES,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
print(json.dumps(repaired_rve.report['surface_quality_repair'], indent=2))
repaired_rve_surface = pv.PolyData(
    repaired_rve.points,
    np.column_stack((np.full(len(repaired_rve.triangles), 3), repaired_rve.triangles)).ravel())
for key, values in dict(grain_a=repaired_rve.grain_pairs[:, 0],
                        grain_b=repaired_rve.grain_pairs[:, 1],
                        is_rve_cap=repaired_rve.exterior.astype(np.uint8),
                        rve_face=repaired_rve.rve_face).items():
    repaired_rve_surface.cell_data[key] = values
repaired_rve_surface.save(REPAIRED_OUTPUT)


### Raise the smallest surface-triangle angles

`REPAIR_SURFACE_ANGLES` works on every triangle whose smallest angle is below `SURFACE_MIN_ANGLE_DEGREES`, by in-patch diagonal flips, moving nodes along their own surface (patch-interior nodes) or along their own curve (junction lines and RVE-face traces), and merging very short edges. RVE-plane coordinates stay exact; curve corners and frozen grains do not change. Each change must raise the smallest angle of the triangles it touches, keep their orientation, and stay within `SURFACE_ANGLE_MAX_DEVIATION` of the previous surface (`None`: 0.1 x the median edge length). Changes that create intersections or facet openings below `MIN_FACET_OPENING_DEGREES` are undone.

Sliver triangles are the main source of poor tetrahedra. Patch corners sharper than the target, and triangles next to frozen grains, can remain below it; the counts before and after are printed. A larger deviation or more passes removes more, at the cost of a larger shape change or run time.

In [ ]:
REPAIR_SURFACE_ANGLES = True
SURFACE_MIN_ANGLE_DEGREES = 30.0
SURFACE_ANGLE_MAX_PASSES = 12
SURFACE_ANGLE_MAX_DEVIATION = None  # physical units; None = 0.1 x median edge length
WEDGE_NO_SHARPENING_DEGREES = SMOOTH_MIN_WEDGE_DEGREES  # degrees: openings below this may not get sharper (None disables)

_angles = reload(import_module('upxo.meshing.gbconformant.d3v2p1.surface_angles'))
repaired_rve = frozen_geometry.run(_angles.improve_surface_angles,
    repaired_rve, enabled=REPAIR_SURFACE_ANGLES, minimum_angle=SURFACE_MIN_ANGLE_DEGREES,
    max_passes=SURFACE_ANGLE_MAX_PASSES, max_deviation=SURFACE_ANGLE_MAX_DEVIATION,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES, frozen_grain_ids=FROZEN_GRAIN_IDS,
    wedge_limit=WEDGE_NO_SHARPENING_DEGREES)
_angle_report = repaired_rve.report['surface_angle_repair']
print(json.dumps({k: v for k, v in _angle_report.items() if k != 'history'}, indent=2))
repaired_rve_surface = pv.PolyData(
    repaired_rve.points,
    np.column_stack((np.full(len(repaired_rve.triangles), 3), repaired_rve.triangles)).ravel())
for key, values in dict(grain_a=repaired_rve.grain_pairs[:, 0],
                        grain_b=repaired_rve.grain_pairs[:, 1],
                        is_rve_cap=repaired_rve.exterior.astype(np.uint8),
                        rve_face=repaired_rve.rve_face).items():
    repaired_rve_surface.cell_data[key] = values
repaired_rve_surface.save(REPAIRED_OUTPUT)

In [ ]:
# A fresh report is required: the pre-repair report is not reused.
_validation = reload(import_module('upxo.meshing.gbconformant.d3v2p1.tet_validation'))
repaired_validation = _validation.validate_tet_surfaces(
    repaired_rve, expected_grains=np.unique(labels_upscaled),
    min_triangle_quality=MIN_SURFACE_TRIANGLE_QUALITY, check_intersections=True, minimum_facet_angle=MIN_FACET_OPENING_DEGREES)
print('After repair:', repaired_validation['status'])
for category in ('blockers', 'warnings', 'unchecked'):
    print(category.upper(), repaired_validation[category])
print('Surface quality:', repaired_validation.get('triangle_quality'))
REPAIRED_OUTPUT.with_suffix('.json').write_text(json.dumps({
    'repair': repaired_rve.report.get('contact_repair'),
    'surface_quality_repair': repaired_rve.report.get('surface_quality_repair'),
    'surface_angle_repair': repaired_rve.report.get('surface_angle_repair'),
    'intersection_repair': repaired_rve.report.get('intersection_repair'),
    'validation': repaired_validation}, indent=2))
if repaired_validation['blockers']:
    raise RuntimeError('Repair has not resolved all surface defects; inspect repaired_validation.')


In [ ]:
MIN_SMOOTH_BOUNDARY_SHARE = 0.85  # at least this share of grain-boundary area must not be voxel-like
_metrics = reload(import_module('upxo.meshing.gbconformant.d3v2p0.surface_metrics'))
boundary_shape = _metrics.voxel_like_fraction(repaired_rve.points, repaired_rve.triangles, repaired_rve.exterior,
                                              repaired_rve.grain_pairs, FROZEN_GRAIN_IDS)
print(f"Smooth grain-boundary area: {boundary_shape['smooth']:.1%} (target {MIN_SMOOTH_BOUNDARY_SHARE:.0%}); "
      f"voxel-like on frozen-grain interfaces: {boundary_shape['on_given_grains'] or 0:.1%}")
if boundary_shape['smooth'] < MIN_SMOOTH_BOUNDARY_SHARE:
    print('WARNING: grain boundaries are more voxel-like than the target.')

### View the grain-boundary surface mesh

`SHOW_SURFACE_MESH` opens a separate window (the notebook does not wait) with the final grain-boundary surface. `SURFACE_SHOW_EDGES` draws the triangle edges; `SURFACE_INTERNAL_ONLY` hides the RVE faces; `SURFACE_COLOR_BY` is `'grain'`, `'min_angle'` (smallest triangle angle) or `'none'`.

In [ ]:
SHOW_SURFACE_MESH = False
SURFACE_SHOW_EDGES = True
SURFACE_INTERNAL_ONLY = True
SURFACE_COLOR_BY = 'min_angle'  # 'grain', 'min_angle' or 'none'

if SHOW_SURFACE_MESH:
    _quality_plots = reload(import_module('upxo.meshing.gbconformant.d3v2p0.quality_plots'))
    surface_view = _quality_plots.launch_surface_view(
        REPAIRED_OUTPUT, show_edges=SURFACE_SHOW_EDGES, internal_only=SURFACE_INTERNAL_ONLY,
        color_by=SURFACE_COLOR_BY, title='CM04_dev: grain-boundary surface')

### Surface gate before tet meshing

Tet quality depends on the surface triangles. If any surface triangle has an angle below `SURFACE_MIN_ANGLE_DEGREES` (set in the surface-angle repair cell, default 30), this cell prints a warning and stops, unless `APPROVE_TET_MESHING_WITH_POOR_SURFACE` is True.

In [ ]:
APPROVE_TET_MESHING_WITH_POOR_SURFACE = True  # set True to mesh anyway after reading the warning

frozen_geometry.check(repaired_rve)
_tri_angles = reload(import_module('upxo.meshing.gbconformant.d3v2p1.surface_angles')).triangle_angles(
    repaired_rve.points, repaired_rve.triangles).min(axis=1)
_poor = int(np.sum(_tri_angles < SURFACE_MIN_ANGLE_DEGREES))
print(f'Surface triangles: smallest angle {_tri_angles.min():.1f} deg; {_poor} of {len(_tri_angles)} below {SURFACE_MIN_ANGLE_DEGREES:g} deg')
if _poor:
    print(f'WARNING: {_poor} surface triangles have an angle below {SURFACE_MIN_ANGLE_DEGREES:g} deg. The surface may be '
          'inadequate for a good-quality tet mesh.')
    if not APPROVE_TET_MESHING_WITH_POOR_SURFACE:
        raise RuntimeError('Tet meshing stopped: set APPROVE_TET_MESHING_WITH_POOR_SURFACE = True to proceed.')

### Generate and verify tetrahedra

Readiness is reported, not enforced: it needs Gmsh minSICN at or above `MIN_TET_QUALITY` and every dihedral angle at or above `MIN_TET_DIHEDRAL_DEGREES` (default 15).

In [ ]:
TET_MESH_SIZE = 1.5 * float(np.min(SPACING))
TET_OPTIMIZE_NETGEN = True  # Keep baseline if optimization worsens a volume
MIN_TET_QUALITY = 0.05  # Gmsh minSICN: larger positive values are better
TET_NETGEN_ALL_VOLUMES = True  # Netgen on every grain, not only grains below MIN_TET_QUALITY
TET_OPTIMIZE_THRESHOLD = 0.5  # Gmsh optimizes tets below this quality; None = Gmsh default
MIN_TET_DIHEDRAL_DEGREES = 15.0  # degrees: smallest acceptable tet dihedral angle (reported)

reload(import_module('upxo.meshing.gbconformant.d3v2p1.tet_validation'))
_tets = reload(import_module('upxo.meshing.gbconformant.d3v2p1.gmsh_tets'))
verified_tets = _tets.mesh_repaired_rve_gmsh(
    repaired_rve, mesh_size=TET_MESH_SIZE, optimize_netgen=TET_OPTIMIZE_NETGEN,
    minimum_quality=MIN_TET_QUALITY, netgen_all_volumes=TET_NETGEN_ALL_VOLUMES,
    optimize_threshold=TET_OPTIMIZE_THRESHOLD, minimum_dihedral=MIN_TET_DIHEDRAL_DEGREES)
tet_mesh_ready = verified_tets.report['ready']
TET_OUTPUT = FINAL_RVE_OUTPUT.with_name('CM04_dev_verified_tets.vtu')
_tets.save_grain_tetrahedra(verified_tets, TET_OUTPUT)
print('Status:', verified_tets.report['status'], '| minimum minSICN:', round(verified_tets.report['minimum_quality'], 4))
print('Dihedral angles:', json.dumps(verified_tets.report['dihedral_angles']))


### Raise tetrahedral dihedral angles

`SMOOTH_TET_DIHEDRALS` moves tet nodes inside grains. With `MOVE_SURFACE_NODES`, grain-surface nodes also move: nodes inside a patch along their surface, junction-line and RVE-trace nodes along their line. RVE-plane coordinates stay exact; curve corners and frozen grains stay fixed.

Each move must raise the worst angle quality of the tets around the node. A neighbouring tet whose smallest dihedral is at or above `TET_TARGET_MIN_DIHEDRAL` may lose angle down to `TET_NEIGHBOUR_FLOOR`, not below; a tet below the target loses none. The largest dihedral follows the same rule against `TET_MAX_DIHEDRAL`. A surface node's move also keeps the same rule on the surrounding surface triangles (`SURFACE_TRIANGLE_TARGET`, `SURFACE_TRIANGLE_FLOOR`), stays within `SURFACE_MOVE_MAX_DEVIATION` of the previous surface, and keeps facet openings at or above `MIN_FACET_OPENING_DEGREES`.

Quality (Gmsh minSICN), dihedral statistics and readiness are recomputed. Each grain's tet volume must equal the volume its moved surface encloses, the total must equal the RVE volume, and moved surface triangles are checked for intersections. The moved surface replaces the saved repaired surface.

In [ ]:
SMOOTH_TET_DIHEDRALS = True
TET_TARGET_MIN_DIHEDRAL = 30.0  # degrees: aspirational smallest tet dihedral angle
TET_MAX_DIHEDRAL = 150.0  # degrees: largest acceptable tet dihedral angle
TET_NEIGHBOUR_FLOOR = 20.0  # degrees: a neighbour at or above the target may fall to this, not below
TET_SMOOTHING_PASSES = 10
MOVE_SURFACE_NODES = True  # grain-surface nodes slide along their surface or line
SURFACE_TRIANGLE_TARGET = 30.0  # degrees: surface triangle angle target during node moves
SURFACE_TRIANGLE_FLOOR = 20.0  # degrees: a surface triangle at or above the target may fall to this
SURFACE_MOVE_MAX_DEVIATION = None  # physical units; None = 0.1 x median surface edge length

_tet_smoothing = reload(import_module('upxo.meshing.gbconformant.d3v2p1.tet_smoothing'))
verified_tets = _tet_smoothing.smooth_grain_tetrahedra(
    verified_tets, len(repaired_rve.points), enabled=SMOOTH_TET_DIHEDRALS,
    target=TET_TARGET_MIN_DIHEDRAL, max_angle=TET_MAX_DIHEDRAL, neighbour_floor=TET_NEIGHBOUR_FLOOR,
    max_passes=TET_SMOOTHING_PASSES, surface=repaired_rve if MOVE_SURFACE_NODES else None,
    frozen_grain_ids=FROZEN_GRAIN_IDS, max_deviation=SURFACE_MOVE_MAX_DEVIATION,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES, triangle_target=SURFACE_TRIANGLE_TARGET,
    triangle_floor=SURFACE_TRIANGLE_FLOOR,
    wedge_limit=WEDGE_NO_SHARPENING_DEGREES)
if MOVE_SURFACE_NODES:
    repaired_rve = _tet_smoothing.surface_after_smoothing(repaired_rve, verified_tets)
    frozen_geometry.check(repaired_rve)
    repaired_rve_surface.points = repaired_rve.points
    repaired_rve_surface.save(REPAIRED_OUTPUT)
tet_mesh_ready = verified_tets.report['ready']
_tets.save_grain_tetrahedra(verified_tets, TET_OUTPUT)
print('Status:', verified_tets.report['status'], '| minimum minSICN:', round(verified_tets.report['minimum_quality'], 4))
print('Dihedral angles:', json.dumps(verified_tets.report['dihedral_angles']))


### Swap faces and edges of poor tetrahedra

`SWAP_TETS` applies 2-3 face swaps and 3-2 edge swaps to tets outside the dihedral limits. Only tets of one grain take part and never a face or edge on a grain surface, so grain surfaces, conformity and grain volumes do not change. A swap is kept only when the worst tet involved improves, the number of tets below `TET_TARGET_MIN_DIHEDRAL` does not rise, and the new tets fill exactly the old volume. `SMOOTH_AFTER_SWAPS` then repeats the node smoothing with the same settings, since swaps and node moves fix different slivers. Conformity, volumes, quality and readiness are rechecked.

In [ ]:
SWAP_TETS = True
TET_SWAP_PASSES = 5

_tet_swaps = reload(import_module('upxo.meshing.gbconformant.d3v2p1.tet_swaps'))
verified_tets = _tet_swaps.swap_grain_tetrahedra(
    verified_tets, repaired_rve, enabled=SWAP_TETS, target=TET_TARGET_MIN_DIHEDRAL,
    max_angle=TET_MAX_DIHEDRAL, max_passes=TET_SWAP_PASSES)
tet_mesh_ready = verified_tets.report['ready']
_tets.save_grain_tetrahedra(verified_tets, TET_OUTPUT)
print('Status:', verified_tets.report['status'], '| minimum minSICN:', round(verified_tets.report['minimum_quality'], 4))
print('Dihedral angles:', json.dumps(verified_tets.report['dihedral_angles']))


### Smooth again after swaps

Swaps and node moves fix different slivers, so node smoothing runs again with the settings of the smoothing cell above. `SMOOTH_AFTER_SWAPS` switches it off.

In [ ]:
SMOOTH_AFTER_SWAPS = True

verified_tets = _tet_smoothing.smooth_grain_tetrahedra(
    verified_tets, len(repaired_rve.points), enabled=SMOOTH_AFTER_SWAPS and SMOOTH_TET_DIHEDRALS,
    target=TET_TARGET_MIN_DIHEDRAL, max_angle=TET_MAX_DIHEDRAL, neighbour_floor=TET_NEIGHBOUR_FLOOR,
    max_passes=TET_SMOOTHING_PASSES, surface=repaired_rve if MOVE_SURFACE_NODES else None,
    frozen_grain_ids=FROZEN_GRAIN_IDS, max_deviation=SURFACE_MOVE_MAX_DEVIATION,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES, triangle_target=SURFACE_TRIANGLE_TARGET,
    triangle_floor=SURFACE_TRIANGLE_FLOOR,
    wedge_limit=WEDGE_NO_SHARPENING_DEGREES)
if MOVE_SURFACE_NODES:
    repaired_rve = _tet_smoothing.surface_after_smoothing(repaired_rve, verified_tets)
    frozen_geometry.check(repaired_rve)
    repaired_rve_surface.points = repaired_rve.points
    repaired_rve_surface.save(REPAIRED_OUTPUT)
tet_mesh_ready = verified_tets.report['ready']
_tets.save_grain_tetrahedra(verified_tets, TET_OUTPUT)
print('Status:', verified_tets.report['status'], '| minimum minSICN:', round(verified_tets.report['minimum_quality'], 4))
print('Dihedral angles:', json.dumps(verified_tets.report['dihedral_angles']))


### Final pass at the dihedral gate

`GATE_PASS` repeats node smoothing and swaps with the target set to `MIN_TET_DIHEDRAL_DEGREES`: tets at or above the gate may change freely but never fall below it, tets below it never get worse, and the worst tet around each moved node must improve. This can only reduce the number of tets below the gate (the count below `TET_TARGET_MIN_DIHEDRAL` may rise slightly). `JUNCTION_POINT_MAX_DEVIATION` also lets junction points (where three or more junction lines meet) move within that distance of the surface; RVE corners and frozen grains stay fixed; `None` keeps junction points fixed. Run the gate smoothing and gate swap cells again for another round.

`INSERT_NODES` then works on tets still below the gate: the tets around their interior nodes (or a ring of same-grain neighbours) are replaced by tets joined to one new, optimally placed node. Grain surfaces stay; a change is kept only if the new tets fill exactly the old volume, the worst tet improves, and the number of tets outside the limits does not rise.

In [ ]:
GATE_PASS = True
JUNCTION_POINT_MAX_DEVIATION = 0.15 * float(np.min(SPACING))  # physical units; None keeps junction points fixed
GATE_TRIANGLE_TARGET = 20.0  # degrees: surface triangle angle target during the gate pass
GATE_TRIANGLE_FLOOR = 15.0  # degrees: a surface triangle at or above the gate target may fall to this

verified_tets = _tet_smoothing.smooth_grain_tetrahedra(
    verified_tets, len(repaired_rve.points), enabled=GATE_PASS, target=MIN_TET_DIHEDRAL_DEGREES,
    max_angle=TET_MAX_DIHEDRAL, neighbour_floor=MIN_TET_DIHEDRAL_DEGREES,
    max_passes=TET_SMOOTHING_PASSES, surface=repaired_rve if MOVE_SURFACE_NODES else None,
    frozen_grain_ids=FROZEN_GRAIN_IDS, max_deviation=SURFACE_MOVE_MAX_DEVIATION,
    minimum_facet_angle=MIN_FACET_OPENING_DEGREES, triangle_target=GATE_TRIANGLE_TARGET,
    triangle_floor=GATE_TRIANGLE_FLOOR,
    corner_max_deviation=JUNCTION_POINT_MAX_DEVIATION if MOVE_SURFACE_NODES else None,
    wedge_limit=WEDGE_NO_SHARPENING_DEGREES)
if MOVE_SURFACE_NODES:
    repaired_rve = _tet_smoothing.surface_after_smoothing(repaired_rve, verified_tets)
    frozen_geometry.check(repaired_rve)
    repaired_rve_surface.points = repaired_rve.points
    repaired_rve_surface.save(REPAIRED_OUTPUT)
tet_mesh_ready = verified_tets.report['ready']
_tets.save_grain_tetrahedra(verified_tets, TET_OUTPUT)
print('Status:', verified_tets.report['status'], '| minimum minSICN:', round(verified_tets.report['minimum_quality'], 4))
print('Dihedral angles:', json.dumps(verified_tets.report['dihedral_angles']))


### Gate swaps

Face and edge swaps with the target set to `MIN_TET_DIHEDRAL_DEGREES`.

In [ ]:
verified_tets = _tet_swaps.swap_grain_tetrahedra(
    verified_tets, repaired_rve, enabled=GATE_PASS, target=MIN_TET_DIHEDRAL_DEGREES,
    max_angle=TET_MAX_DIHEDRAL, max_passes=TET_SWAP_PASSES)
tet_mesh_ready = verified_tets.report['ready']
_tets.save_grain_tetrahedra(verified_tets, TET_OUTPUT)
print('Status:', verified_tets.report['status'], '| minimum minSICN:', round(verified_tets.report['minimum_quality'], 4))
print('Dihedral angles:', json.dumps(verified_tets.report['dihedral_angles']))


### Node insertion

`INSERT_NODES` works on tets still below `MIN_TET_DIHEDRAL_DEGREES` or above `TET_MAX_DIHEDRAL`: the tets around their interior nodes, around one of their interior edges, at one of their grain-surface edges, or a ring of same-grain neighbours are replaced by tets joined to one new, optimally placed node. Grain surfaces stay; a change is kept only if the new tets fill exactly the old volume, the worst tet improves and the number of tets outside the limits does not rise.

In [ ]:
INSERT_NODES = True

verified_tets = _tet_swaps.insert_grain_tetrahedra(
    verified_tets, repaired_rve, enabled=INSERT_NODES, target=MIN_TET_DIHEDRAL_DEGREES,
    max_angle=TET_MAX_DIHEDRAL)
print('Node insertion:', json.dumps({k: v for k, v in verified_tets.report['node_insertion'].items() if k != 'history'}))
tet_mesh_ready = verified_tets.report['ready']
_tets.save_grain_tetrahedra(verified_tets, TET_OUTPUT)
print('Status:', verified_tets.report['status'], '| minimum minSICN:', round(verified_tets.report['minimum_quality'], 4))
print('Dihedral angles:', json.dumps(verified_tets.report['dihedral_angles']))


## Export tetrahedra only to Abaqus

Run after tetrahedral verification. Writes a master deck, numbered node/element/grain-set includes, README and mesh statistics. `ABAQUS_ELSET_PREFIX` defaults to `grn_`; sets use retained grain IDs. `ABAQUS_UNITS` documents the existing coordinate units (no conversion). Each run creates a new timestamped folder. No surface elements or analysis definitions are exported.

In [ ]:
from datetime import datetime
from importlib import import_module, reload

ABAQUS_ELSET_PREFIX = 'grn_'
ABAQUS_UNITS = 'unspecified'  # Set to the units used for SPACING; no coordinate scaling.
ABAQUS_OUTPUT = RUN_DIR / 'abaqus' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')

REQUIRE_READY_TET_MESH = False  # True refuses to export a mesh that missed the tet quality thresholds

if globals().get('verified_tets') is None:
    raise RuntimeError('Run the tetrahedral verification cell first.')
if REQUIRE_READY_TET_MESH and not tet_mesh_ready:
    raise RuntimeError('Tet mesh is below the quality thresholds; inspect verified_tets.report before exporting.')
_abaqus = reload(import_module('upxo.meshing.gbconformant.d3v2p0.abaqus_export'))
abaqus_folder = _abaqus.export_tets_abaqus(
    verified_tets, ABAQUS_OUTPUT, prefix=ABAQUS_ELSET_PREFIX,
    units=ABAQUS_UNITS, expected_grains=retained_grain_ids)
print('Abaqus mesh:', abaqus_folder / 'model_master.inp')
print('Mesh information:', abaqus_folder / 'README.md')


## Element quality distributions

One histogram per measure (log counts): surface triangles (smallest angle, largest angle, shape quality; grain boundaries and RVE faces separately) and tetrahedra (smallest and largest dihedral angle, Gmsh minSICN). Each plot marks the worst value in the current mesh (solid red line) and shades the band from it towards the best possible value, with the number of elements inside: [lowest, lowest + x(best - lowest)] for smallest angles and quality, [highest - x(highest - best), highest] for largest angles, where x is `QUALITY_WORST_FRACTION` (default 0.05). Best possible values: 60 deg for triangle angles, 70.53 deg (regular tet) for dihedral angles, 1 for shape quality and minSICN. The limits used above are dashed. `SHOW_QUALITY_DISTRIBUTIONS` switches the plots on.

In [ ]:
SHOW_QUALITY_DISTRIBUTIONS = True
QUALITY_WORST_FRACTION = 0.05  # count elements within this fraction of the way from the worst value to the best possible

if SHOW_QUALITY_DISTRIBUTIONS:
    import matplotlib.pyplot as plt
    _quality_plots = reload(import_module('upxo.meshing.gbconformant.d3v2p0.quality_plots'))
    triangle_figures, triangle_stats = _quality_plots.triangle_quality_distribution(
        repaired_rve.points, repaired_rve.triangles, repaired_rve.exterior,
        min_angle_limit=SURFACE_MIN_ANGLE_DEGREES, fraction=QUALITY_WORST_FRACTION)
    print('Surface triangles:', json.dumps(triangle_stats, indent=2))
    plt.show()

In [ ]:
if SHOW_QUALITY_DISTRIBUTIONS:
    tet_figures, tet_stats = _quality_plots.tet_quality_distribution(
        verified_tets.points, verified_tets.tetrahedra, verified_tets.quality,
        min_dihedral_limit=MIN_TET_DIHEDRAL_DEGREES, max_dihedral_limit=TET_MAX_DIHEDRAL,
        fraction=QUALITY_WORST_FRACTION)
    print('Tetrahedra:', json.dumps(tet_stats, indent=2))
    plt.show()

## Clipped 3D view of the tetrahedra

Opens a PyVista window with the tetrahedra of a set of mutually non-touching grains, cut by a plane. `CLIP_CRINKLE = True` keeps whole tetrahedra intersected by the plane instead of cutting them, so the displayed elements are the real mesh elements. Set `CLIP_NORMAL` (`'x'`, `'y'` or `'z'`) and `CLIP_ORIGIN` to move the plane, and `SELECTION_SEED` to pick another grain set. Junction lines are drawn in red. The cell waits until the window is closed; set `SHOW_TET_VIEW_CLIPPED = False` to skip it. The first cell builds the tetrahedral grid from the final verified tetrahedra.

In [ ]:
# Tet grid for the clipped view (from the final verified tets).
import pyvista as pv
tet_grid = pv.UnstructuredGrid(
    np.column_stack((np.full(len(verified_tets.tetrahedra), 4), verified_tets.tetrahedra)).ravel(),
    np.full(len(verified_tets.tetrahedra), pv.CellType.TETRA, dtype=np.uint8), verified_tets.points)
tet_grid.cell_data['grain_id'] = verified_tets.grain_ids
tet_grid.cell_data['minSICN'] = verified_tets.quality

In [ ]:
SHOW_TET_VIEW_CLIPPED = True

if SHOW_TET_VIEW_CLIPPED:
    from importlib import import_module, reload
    _views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))
    SELECTION_SEED = 42  # Change to select another mutually non-touching set.
    MAX_GRAINS = None  # Positive integer to limit the number displayed.
    COLOR_BY_SEGMENT = True  # False: grain ID; True: disconnected-component segment ID.
    TET_CMAP = 'tab20c'
    TET_OPACITY = 1.0
    TET_SHOW_EDGES = True
    SHOW_JUNCTION_LINES = True
    JUNCTION_LINE_COLOR = 'red'
    JUNCTION_LINE_WIDTH = 3.0
    SHOW_JUNCTION_POINTS = True
    JUNCTION_POINT_COLOR = 'black'
    JUNCTION_POINT_SIZE = 10.0
    INCLUDE_RVE_JUNCTIONS = True
    CLIP_ENABLED = True
    CLIP_CRINKLE = True  # Keep whole intersected tetrahedra.
    CLIP_NORMAL = 'x'
    CLIP_ORIGIN = None  # None uses the selected mesh centre; or specify (x, y, z).
    CLIP_INVERT = False

    _tet_view, displayed_grain_ids = _views.tet_view(
        repaired_rve, tet_grid, seed=SELECTION_SEED, maximum=MAX_GRAINS,
        color_segments=COLOR_BY_SEGMENT, cmap=TET_CMAP, opacity=TET_OPACITY,
        show_edges=TET_SHOW_EDGES,
        show_lines=SHOW_JUNCTION_LINES, line_color=JUNCTION_LINE_COLOR,
        line_width=JUNCTION_LINE_WIDTH, show_points=SHOW_JUNCTION_POINTS,
        point_color=JUNCTION_POINT_COLOR, point_size=JUNCTION_POINT_SIZE,
        include_rve_junctions=INCLUDE_RVE_JUNCTIONS,
        clip=CLIP_ENABLED, crinkle=CLIP_CRINKLE, normal=CLIP_NORMAL,
        origin=CLIP_ORIGIN, invert=CLIP_INVERT)

## Clipped 3D view of all grains

The same clipped view for every grain at once, with the same options. Each grain is coloured by its grain ID and junction lines are drawn in red. `ALL_CLIP_CRINKLE = True` keeps whole tetrahedra cut by the plane. The cell waits until the window is closed; set `SHOW_ALL_GRAINS_CLIPPED = False` to skip it.

In [ ]:
SHOW_ALL_GRAINS_CLIPPED = True

if SHOW_ALL_GRAINS_CLIPPED:
    from importlib import import_module, reload
    import pyvista as pv
    _views = reload(import_module('upxo.meshing.gbconformant.d3v2p0.views'))
    ALL_TET_CMAP = 'nipy_spectral'
    ALL_TET_OPACITY = 1.0
    ALL_TET_SHOW_EDGES = True
    ALL_SHOW_JUNCTION_LINES = True
    ALL_JUNCTION_LINE_COLOR = 'red'
    ALL_JUNCTION_LINE_WIDTH = 3.0
    ALL_SHOW_JUNCTION_POINTS = False
    ALL_JUNCTION_POINT_COLOR = 'black'
    ALL_JUNCTION_POINT_SIZE = 10.0
    ALL_INCLUDE_RVE_JUNCTIONS = True
    ALL_CLIP_ENABLED = True
    ALL_CLIP_CRINKLE = True  # Keep whole intersected tetrahedra.
    ALL_CLIP_NORMAL = 'x'
    ALL_CLIP_ORIGIN = None  # None uses the mesh centre; or specify (x, y, z).
    ALL_CLIP_INVERT = False

    _plotter = pv.Plotter(notebook=False)
    _mesh_kwargs = dict(scalars='grain_id', cmap=ALL_TET_CMAP, opacity=ALL_TET_OPACITY,
                        show_edges=ALL_TET_SHOW_EDGES)
    if ALL_CLIP_ENABLED:
        _plotter.add_mesh_clip_plane(tet_grid, crinkle=ALL_CLIP_CRINKLE, normal=ALL_CLIP_NORMAL,
                                     origin=ALL_CLIP_ORIGIN, invert=ALL_CLIP_INVERT, **_mesh_kwargs)
    else:
        _plotter.add_mesh(tet_grid, **_mesh_kwargs)
    if ALL_SHOW_JUNCTION_LINES or ALL_SHOW_JUNCTION_POINTS:
        _lines, _points = _views.junction_geometry(repaired_rve, None, ALL_INCLUDE_RVE_JUNCTIONS)
        if ALL_SHOW_JUNCTION_LINES and _lines.n_lines:
            _plotter.add_mesh(_lines, color=ALL_JUNCTION_LINE_COLOR, line_width=ALL_JUNCTION_LINE_WIDTH)
        if ALL_SHOW_JUNCTION_POINTS and len(_points):
            _plotter.add_points(_points, color=ALL_JUNCTION_POINT_COLOR, point_size=ALL_JUNCTION_POINT_SIZE,
                                render_points_as_spheres=True)
    _plotter.add_axes()
    print(f'{len(np.unique(verified_tets.grain_ids))} grains; {tet_grid.n_cells:,} tetrahedra')
    _plotter.show()